In [1]:
from langchain_community.document_loaders import BiliBiliLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [4]:
video_urls = [
    "https://www.bilibili.com/video/BV1Bo4y1A7FU", 
    "https://www.bilibili.com/video/BV1ug4y157xA",
    "https://www.bilibili.com/video/BV1yh411V7ge",
    "https://www.bilibili.com/video/BV1MxfcBhEdo/?spm_id_from=333.1007.tianma.1-1-1.click&vd_source=f1552727c1557945d8c7a13ea5b929ef"
]

bili = []

video_loader = BiliBiliLoader(video_urls)

In [5]:
video_docs = video_loader.load()
len(video_docs), video_docs[0].metadata.keys()

(4,
 dict_keys(['bvid', 'aid', 'videos', 'tid', 'tid_v2', 'tname', 'tname_v2', 'copyright', 'pic', 'title', 'pubdate', 'ctime', 'desc', 'desc_v2', 'state', 'duration', 'rights', 'owner', 'stat', 'argue_info', 'dynamic', 'cid', 'dimension', 'season_id', 'premiere', 'teenage_mode', 'is_chargeable_season', 'is_story', 'is_upower_exclusive', 'is_upower_play', 'is_upower_preview', 'enable_vt', 'vt_display', 'is_upower_exclusive_with_qa', 'no_cache', 'pages', 'subtitle', 'ugc_season', 'is_season_display', 'user_garb', 'honor_reply', 'like_icon', 'need_jump_bv', 'disable_show_up_info', 'is_story_play', 'is_view_self', 'url']))

In [6]:
bili = []
for doc in video_docs:
    original = doc.metadata

    metadata = {
            'title': original.get('title', '未知标题'),
            'author': original.get('owner', {}).get('name', '未知作者'),
            'source': original.get('bvid', '未知ID'),
            'view_count': original.get('stat', {}).get('view', 0),
            'length': original.get('duration', 0),
    }
    
    doc.metadata = metadata
    bili.append(doc)
bili[0].metadata.keys()

dict_keys(['title', 'author', 'source', 'view_count', 'length'])

In [7]:
embed_model = HuggingFaceEmbeddings(model='BAAI/bge-small-zh-v1.5')
vector_store = Chroma.from_documents(bili, embed_model)

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.chains.query_constructor.base import AttributeInfo
from langchain_community.chat_models.moonshot import MoonshotChat
import os

In [9]:
llm = MoonshotChat(
    model="kimi-k2-0905-preview", 
    temperature=0, 
    max_tokens=1024, 
    api_key=os.getenv("MOONSHOT_API_KEY"), 
)

In [11]:
# 3. 配置元数据字段信息
metadata_field_info = [
    AttributeInfo(
        name="title",
        description="视频标题（字符串）",
        type="string", 
    ),
    AttributeInfo(
        name="author",
        description="视频作者（字符串）",
        type="string",
    ),
    AttributeInfo(
        name="view_count",
        description="视频观看次数（整数）",
        type="integer",
    ),
    AttributeInfo(
        name="length",
        description="视频长度，以秒为单位的整数",
        type="integer"
    )
]

In [12]:
retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vector_store,
    document_contents="记录视频标题、作者、观看次数等信息的视频元数据", 
    metadata_field_info=metadata_field_info, 
    enable_limit=True,  # 允许大模型自己决定返回数量
    verbose=True,  # 在控制台打印出模型到底生成了什么查询
)

In [22]:
queries = [
    "时间最短的视频",
    "时间最长的视频", 
    "时长大于600秒的视频", 
    "观看次数小于五万的视频"
]

In [27]:
for query in queries:
    print(f"\n--- 查询: '{query}' ---")
    results = retriever.invoke(query)
    if results:
        for doc in results:
            title = doc.metadata.get('title', '未知标题')
            author = doc.metadata.get('author', '未知作者')
            view_count = doc.metadata.get('view_count', '未知')
            length = doc.metadata.get('length', '未知')
            print(f"标题: {title}")
            print(f"作者: {author}")
            print(f"观看次数: {view_count}")
            print(f"时长: {length}秒")
            print("="*50)
    else:
        print("未找到匹配的视频")


--- 查询: '时间最短的视频' ---
标题: 黄金白银大崩盘，谁是幕后推手？
作者: 小Lin说
观看次数: 1687047
时长: 1176秒

--- 查询: '时间最长的视频' ---
标题: 黄金白银大崩盘，谁是幕后推手？
作者: 小Lin说
观看次数: 1687047
时长: 1176秒

--- 查询: '时长大于600秒的视频' ---
标题: 《吴恩达 x OpenAI Prompt课程》【专业翻译，配套代码笔记】02.Prompt 的构建原则
作者: 二次元的Datawhale
观看次数: 20235
时长: 1063秒
标题: 《吴恩达 x OpenAI Prompt课程》【专业翻译，配套代码笔记】03.Prompt如何迭代优化
作者: 二次元的Datawhale
观看次数: 7670
时长: 806秒
标题: 黄金白银大崩盘，谁是幕后推手？
作者: 小Lin说
观看次数: 1687047
时长: 1176秒

--- 查询: '观看次数小于五万的视频' ---
标题: 《吴恩达 x OpenAI Prompt课程》【专业翻译，配套代码笔记】01.课程介绍
作者: 二次元的Datawhale
观看次数: 48967
时长: 390秒
标题: 《吴恩达 x OpenAI Prompt课程》【专业翻译，配套代码笔记】02.Prompt 的构建原则
作者: 二次元的Datawhale
观看次数: 20235
时长: 1063秒
标题: 《吴恩达 x OpenAI Prompt课程》【专业翻译，配套代码笔记】03.Prompt如何迭代优化
作者: 二次元的Datawhale
观看次数: 7670
时长: 806秒


In [26]:
for i in bili:
    print(i.metadata)

{'title': '《吴恩达 x OpenAI Prompt课程》【专业翻译，配套代码笔记】01.课程介绍', 'author': '二次元的Datawhale', 'source': 'BV1Bo4y1A7FU', 'view_count': 48967, 'length': 390}
{'title': '《吴恩达 x OpenAI Prompt课程》【专业翻译，配套代码笔记】02.Prompt 的构建原则', 'author': '二次元的Datawhale', 'source': 'BV1ug4y157xA', 'view_count': 20235, 'length': 1063}
{'title': '《吴恩达 x OpenAI Prompt课程》【专业翻译，配套代码笔记】03.Prompt如何迭代优化', 'author': '二次元的Datawhale', 'source': 'BV1yh411V7ge', 'view_count': 7670, 'length': 806}
{'title': '黄金白银大崩盘，谁是幕后推手？', 'author': '小Lin说', 'source': 'BV1MxfcBhEdo', 'view_count': 1687047, 'length': 1176}
